In [5]:
# first install libraries and create mongoDB connection 
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [6]:

#the i'll import pymongo and datetime lib
from datetime import datetime, timezone
from pymongo import MongoClient

print("Imports successful!")

Imports successful!


In [7]:
from pymongo import MongoClient

# Connect Python to local MongoDB
client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

# Check connection
try:
    client.admin.command("ping")
    print("MongoDB connected successfully!")
except Exception as e:
    print("Connection failed:", e)

MongoDB connected successfully!


In [8]:
# Create database
db = client["banking_db"]

# Create collections
customers = db["customers"]
accounts = db["accounts"]
transactions = db["transactions"]

print("Database:", db.name)
print("Collections:", db.list_collection_names())

Database: banking_db
Collections: ['accounts', 'customers', 'transactions']


In [9]:
from datetime import datetime, timezone

# Sample customers
customer_data = [
    {
        "customer_id": "C101",
        "name": "Aarav Sharma",
        "city": "Pune",
        "age": 25
    },
    {
        "customer_id": "C102",
        "name": "Priya Patil",
        "city": "Mumbai",
        "age": 30
    },
    {
        "customer_id": "C103",
        "name": "Rahul Verma",
        "city": "Nagpur",
        "age": 28
    }
]

# Insert customers only if these IDs don't exist
for customer in customer_data:
    customers.update_one(
        {"customer_id": customer["customer_id"]},
        {"$setOnInsert": customer},
        upsert=True
    )

print("Customer data inserted!")

Customer data inserted!


In [5]:
account_data = [
    {
        "account_no": "ACC1001",
        "customer_id": "C101",
        "account_type": "Savings",
        "balance": 15000.0,
        "status": "Active"
    },
    {
        "account_no": "ACC1002",
        "customer_id": "C102",
        "account_type": "Current",
        "balance": 25000.0,
        "status": "Active"
    },
    {
        "account_no": "ACC1003",
        "customer_id": "C103",
        "account_type": "Savings",
        "balance": 8000.0,
        "status": "Active"
    }
]

for account in account_data:
    accounts.update_one(
        {"account_no": account["account_no"]},
        {"$setOnInsert": account},
        upsert=True
    )

print("Account data inserted!")

Account data inserted!


In [10]:

#check records 

for account in accounts.find({}, {"_id": 0}):
    print(account)

{'account_no': 'ACC1001', 'account_type': 'Savings', 'balance': 14000.0, 'customer_id': 'C101', 'status': 'Active'}
{'account_no': 'ACC1002', 'account_type': 'Current', 'balance': 25000.0, 'customer_id': 'C102', 'status': 'Active'}
{'account_no': 'ACC1003', 'account_type': 'Savings', 'balance': 8000.0, 'customer_id': 'C103', 'status': 'Active'}


In [11]:
# read customer information

for customer in customers.find({}, {"_id": 0}):
    print(customer)

{'customer_id': 'C101', 'age': 25, 'city': 'Yavatmal', 'name': 'Aarav Sharma'}
{'customer_id': 'C102', 'age': 30, 'city': 'Mumbai', 'name': 'Priya Patil'}
{'customer_id': 'C103', 'age': 28, 'city': 'Nagpur', 'name': 'Rahul Verma'}


In [9]:
#find specific customer ID

customer = customers.find_one({"customer_id" : "C101"})
print(customer)

{'_id': ObjectId('6adf6204be895f77ad7beb3c'), 'customer_id': 'C101', 'age': 25, 'city': 'Pune', 'name': 'Aarav Sharma'}


In [14]:
#find only customer from pune 

for customer in customers.find(
    {"city": "Pune"},
    {"_id": 0}
):
    print(customer)

{'customer_id': 'C101', 'age': 25, 'city': 'Pune', 'name': 'Aarav Sharma'}


In [15]:
#find accounts with greater than 10,000
for accounts in accounts.find(
    {"balance" :{"$gt" : 10000}},
    {"_id" : 0}
):
    print(account)

{'account_no': 'ACC1003', 'account_type': 'Savings', 'balance': 8000.0, 'customer_id': 'C103', 'status': 'Active'}
{'account_no': 'ACC1003', 'account_type': 'Savings', 'balance': 8000.0, 'customer_id': 'C103', 'status': 'Active'}


In [16]:
# update customer details (UPDATE)

result = customers.update_one(
    {"customer_id" : "C101"},
    {"$set" : {"city" : "Yavatmal"}}
)
if result.modified_count > 0:
    print("Customer city updated successfully !")
else:
    print("Customer not found or city unchanged.")

Customer city updated successfully !


In [17]:
# verify the update 
print(customers.find_one(
    {"customer_id": "C101"},
    {"_id": 0}
))

{'customer_id': 'C101', 'age': 25, 'city': 'Yavatmal', 'name': 'Aarav Sharma'}


In [3]:
from datetime import datetime, timezone
from pymongo import MongoClient

print("Imports successful!")

Imports successful!


In [4]:
client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

client.admin.command("ping")

db = client["banking_db"]
accounts = db["accounts"]
transactions = db["transactions"]

print("MongoDB connected successfully!")

MongoDB connected successfully!


In [12]:
# deposite accounts function 

def deposit(account_no, amount):

    # Validate deposit amount
    if amount <= 0:
        print("Amount must be greater than zero.")
        return

    # Find active account and update balance
    result = accounts.update_one(
        {
            "account_no": account_no,
            "status": "Active"
        },
        {
            "$inc": {"balance": amount}
        }
    )

    # Check whether account exists
    if result.matched_count == 0:
        print("Active account not found.")
        return

    # Save transaction history
    transactions.insert_one({
        "account_no": account_no,
        "transaction_type": "Deposit",
        "amount": amount,
        "transaction_date": datetime.now(timezone.utc),
        "status": "Success"
    })

    print("Deposit successful!")


# Test the function
deposit("ACC1001", 2000)

Deposit successful!


In [13]:
#verify the update balance
account = accounts.find_one(
    {"account_no" : "ACC1001"},
    {"_id" :0}
)
print(account)

{'account_no': 'ACC1001', 'account_type': 'Savings', 'balance': 16000.0, 'customer_id': 'C101', 'status': 'Active'}


In [14]:
# create withdrwal function
def withdraw(account_no, amount):
    if amount <= 0:
        print("Amount must be greater than zero.")
        return

    result = accounts.update_one(
        {
            "account_no": account_no,
            "status": "Active",
            "balance": {"$gte": amount}
        },
        {"$inc": {"balance": -amount}}
    )

    if result.matched_count == 0:
        print(
            "Withdrawal failed: account unavailable "
            "or insufficient balance."
        )
        return

    transactions.insert_one({
        "account_no": account_no,
        "transaction_type": "Withdrawal",
        "amount": amount,
        "transaction_date": datetime.now(timezone.utc),
        "status": "Success"
    })

    print("Withdrawal successful!")

withdraw("ACC1001", 3000)

Withdrawal successful!


In [17]:
#test the unsuccessful withdrawal
withdraw("ACC1001", 50000)

Withdrawal failed: account unavailable or insufficient balance.


In [18]:
#view check transaction histiry

for transaction in transactions.find(
    {"account_no":"ACC1001"},
    {"_id":0}
).sort("transation_date", -1):
    print(transaction)

{'account_no': 'ACC1001', 'transaction_type': 'Deposit', 'amount': 2000, 'transaction_date': datetime.datetime(2026, 10, 26, 14, 50, 1, 511000), 'status': 'Success'}
{'account_no': 'ACC1001', 'transaction_type': 'Withdrawal', 'amount': 3000, 'transaction_date': datetime.datetime(2026, 10, 26, 14, 56, 43, 76000), 'status': 'Success'}
{'account_no': 'ACC1001', 'transaction_type': 'Deposit', 'amount': 2000, 'transaction_date': datetime.datetime(2026, 10, 26, 15, 54, 19, 571000), 'status': 'Success'}
{'account_no': 'ACC1001', 'transaction_type': 'Withdrawal', 'amount': 3000, 'transaction_date': datetime.datetime(2026, 10, 26, 15, 54, 27, 730000), 'status': 'Success'}


In [19]:
# this is calculate the current account balance


account = accounts.find_one(
    {"account_no": "ACC1001"},
    {"_id": 0}
)

if account:
    print("Account number:", account["account_no"])
    print("Current balance: ₹", account["balance"])

Account number: ACC1001
Current balance: ₹ 13000.0


In [21]:
#total balance across all accounts

pipeline =[
    {
        "$group":{
            "_id": None,
            "total_balance":{"$sum":"$balance"},
            "number_of_accounts":{"$sum": 1}
        }
    }
]

result = list(accounts.aggregate(pipeline))

for item in result:
    print("Total bank account balance:", item["total_balance"])
    print("Number of account :", item["number_of_accounts"])
    

Total bank account balance: 46000.0
Number of account : 3


In [22]:
# now deposit and withdrawal

pipeline = [
    {
        "$match": {"status": "Success"}
    },
    {
        "$group": {
            "_id": "$transaction_type",
            "total_amount": {"$sum": "$amount"},
            "transaction_count": {"$sum": 1}
        }
    }
]

for item in transactions.aggregate(pipeline):
    print(item)

{'_id': 'Deposit', 'total_amount': 4000, 'transaction_count': 2}
{'_id': 'Withdrawal', 'total_amount': 6000, 'transaction_count': 2}


In [23]:
#delete a customer record

result = customers.delete_one(
    {"customer_id": "C103"}
)

if result.deleted_count > 0:
    print("Customer deleted successfully.")
else:
    print("Customer not found.")

Customer deleted successfully.


In [24]:

# verify the customer id in this dataset
print(customers.find_one({"customer_id": "C103"}))

None
